# Model Maxxing — Hito B: Theseus (Coder-7B − 7B → Qwen2.5-1.5B) en Kaggle

**Cómo ejecutarlo:** Settings → Accelerator = **GPU T4 x2**, Internet = **On**. **Save Version → Save & Run All**.
Al terminar descarga **`hito_b_theseus_results.zip`** de Output.

- Estimación (no medida): ~2,5–3 h. Copias intermedias: `hito_b_after_dev.zip`; si falla, `hito_b_theseus_FALLO_*.zip`.
- Descarga ~33,6 GB (7B, Coder-7B, 1.5B) a `/tmp`; los 7B se borran tras `align` y `slices`.
- Pre-registro: `configs/hito_b_theseus.yaml`. α se elige en dev (NLL → pass@1) y se **congela antes del test**.
- Progreso de cada fase paralela: una línea por GPU cada 5 min. Cualquier fallo detiene el notebook.

In [ ]:
!nvidia-smi
%cd /tmp
!rm -rf modelmaxxing-prev && git clone -q -b hito-b-theseus https://github.com/RubenGGBC/modelmaxxing-prev.git
%cd /tmp/modelmaxxing-prev
!git log --oneline -1
!pip install -q -e . "transformers==5.22.2" "datasets==5.1.0" psutil pyyaml 2>&1 | tail -3

In [ ]:
import os, subprocess, time, torch
os.chdir("/tmp/modelmaxxing-prev")
os.environ.update(PYTHONUNBUFFERED="1", PYTHONPATH="src",   # logs en vivo (sin búfer)
                   MM_SANDBOX="subprocess", PYTHONIOENCODING="utf-8", TOKENIZERS_PARALLELISM="false",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")   # menos fragmentación de VRAM
CFG = "configs/hito_b_theseus.yaml"
NOISE = "Warn|Loading weights|it/s|Fetching|Redirects|cached version|Found the latest|  capa [0-9]+/"
N_GPU = max(torch.cuda.device_count(), 1)
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


def sh(cmd, quiet=True):
    """Ejecuta en bash con pipefail y DETIENE el notebook si el comando falla.
    El filtro de ruido nunca provoca fallo (grep sin coincidencias devuelve 1)."""
    if quiet:
        cmd = f"{cmd} 2>&1 | {{ grep -v -E '{NOISE}' || true; }}"
    t0 = time.time()
    r = subprocess.run(["bash", "-o", "pipefail", "-c", cmd])
    print(f"[{time.time() - t0:.0f} s] rc={r.returncode}")
    if r.returncode != 0:
        raise RuntimeError(f"falló (rc={r.returncode}): {cmd}")


def phase(name, extra=""):
    sh(f"python scripts/hito_b_theseus.py {CFG} {name} {extra}")


def variants(stage):
    out = subprocess.run(["python", "scripts/hito_b_theseus.py", CFG, stage, "--list-variants"],
                         capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
    return out.split()


def parallel(stage):
    """Reparte las variantes entre las GPUs: un proceso por GPU (CUDA_VISIBLE_DEVICES)."""
    vs = variants(stage)
    groups = [g for g in (vs[i::N_GPU] for i in range(N_GPU)) if g]
    procs, t0 = [], time.time()
    for g, group in enumerate(groups):
        if g:
            time.sleep(90)   # escalonar: que no carguen el modelo a la vez (RAM de la VM)
        print(f"GPU {g}: {group}")
        log = open(f"/tmp/{stage}_gpu{g}.log", "w")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        procs.append((subprocess.Popen(["python", "scripts/hito_b_theseus.py", CFG, stage, "--only", *group],
                                       stdout=log, stderr=subprocess.STDOUT, env=env), log, g))
    while any(p.poll() is None for p, _, _ in procs):     # progreso cada 5 min
        time.sleep(300)
        for _, _, g in procs:
            done = [l for l in open(f"/tmp/{stage}_gpu{g}.log").read().splitlines()
                    if l.startswith((f"{stage} ", "ya hecho"))]
            print(f"[{time.time() - t0:.0f} s] GPU {g}: {len(done)} variantes terminadas", done[-1:] )
    for p, log, g in procs:
        p.wait()
        log.close()
        print(f"--- GPU {g} (rc={p.returncode}), últimas líneas:")
        print("\n".join(l for l in open(f"/tmp/{stage}_gpu{g}.log").read().splitlines()
                        if "Loading weights" not in l)[-3000:])
    print(f"{stage}: {time.time() - t0:.0f} s")
    bad = [g for p, _, g in procs if p.returncode != 0]
    if bad:
        save_zip(f"hito_b_theseus_FALLO_{stage}.zip")   # conservar parciales y logs antes de parar
        raise RuntimeError(f"{stage} falló en GPU {bad}: ver /tmp/{stage}_gpu*.log (copiados a Output)")


def save_zip(name):
    sh("mkdir -p results/experiments/hito_b_theseus/logs && cp /tmp/*_gpu*.log results/experiments/hito_b_theseus/logs/ "
       "2>/dev/null || true", quiet=False)
    sh(f"zip -qr /kaggle/working/{name} results/experiments/hito_b_theseus -x '*/model.safetensors' "
       f"&& ls -la /kaggle/working", quiet=False)


## 1. Tests y recursos (antes de descargar)

In [ ]:
sh("python -m pytest -q 2>&1 | tail -3", quiet=False)
phase("resources")

## 2. Descarga con revisiones fijadas y nueva comprobación

In [ ]:
sh("python scripts/download_models.py hito_b")
phase("resources")

## 3. Alineamiento Theseus (deltas TH / RAND / NAIVE) y slices de referencia del Hito A

In [ ]:
phase("align")
phase("slices")
sh("rm -rf models_cache/Qwen2.5-7B models_cache/Qwen2.5-Coder-7B && df -h /tmp", quiet=False)

## 4. Desarrollo: NLL de todo el grid → α finalistas → pass@1

In [ ]:
parallel("dev_nll")
phase("pick")
sh("cat results/experiments/hito_b_theseus/dev_pick.json", quiet=False)
parallel("dev_pass")
save_zip("hito_b_after_dev.zip")

## 5. Congelar α (ANTES del test), test (MBPP final + HumanEval+) e informe

In [ ]:
phase("lock")
parallel("test")
phase("report")
save_zip("hito_b_theseus_results.zip")